In [0]:
from delta.tables import DeltaTable

# 1. Configuration (Same as before)
server_name = "car-insurance-db-server.database.windows.net"
database_name = "free-sql-db-7048145"
port = "1433"

user = dbutils.secrets.get(scope="sql-server-cred", key="sql-server-username")
password = dbutils.secrets.get(scope="sql-server-cred", key="sql-server-password")

jdbc_url = f"jdbc:sqlserver://{server_name}:{port};database={database_name}"
connection_properties = {
    "user": user,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# 2. Define tables, targets, AND Primary Keys
# Format: {"source": ("target_table", "storage_path", "primary_key_column")}
tables_to_ingest = {
    "carInsurance.customer": ("`smart-claims-dev`.`01_bronze`.customer", "abfss://bronze@car0claims0datalake.dfs.core.windows.net/customers", "customer_id"),
    "carInsurance.claim": ("`smart-claims-dev`.`01_bronze`.claim", "abfss://bronze@car0claims0datalake.dfs.core.windows.net/claims", "claim_no"),
    "carInsurance.policy": ("`smart-claims-dev`.`01_bronze`.policy", "abfss://bronze@car0claims0datalake.dfs.core.windows.net/policy", "policy_no")
}

# 3. Ingestion & Merge Loop
for sql_table, (databricks_table, storage_path, primary_key) in tables_to_ingest.items():
    print(f"Processing {sql_table}...")
    
    # Read the current state of the Azure SQL table
    df_source = spark.read.jdbc(
        url=jdbc_url,
        table=sql_table,
        properties=connection_properties
    )
    
    # Check if the Delta table already exists in Databricks
    if spark.catalog.tableExists(databricks_table):
        print(f"Table {databricks_table} exists. Performing UPSERT and DELETE...")
        
        # Load the existing Delta table
        delta_table = DeltaTable.forName(spark, databricks_table)
        
        # Build the match condition (e.g., "target.OrderID = source.OrderID")
        merge_condition = f"target.{primary_key} = source.{primary_key}"
        
        # Execute the MERGE
        delta_table.alias("target") \
            .merge(
                df_source.alias("source"),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .whenNotMatchedBySourceDelete() \
            .execute()
            
    else:
        # Table doesn't exist yet, do the initial full load
        print(f"Table {databricks_table} not found. Running initial full load...")
        df_source.write \
            .format("delta") \
            .mode("overwrite") \
            .option("path", storage_path) \
            .saveAsTable(databricks_table)

print("Ingestion, UPSERTS, and DELETES complete!")